In [1]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

import utils.data_processing_bronze_table
import utils.data_processing_silver_table
import utils.data_processing_gold_table


## set up pyspark session

In [2]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/10 13:28:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## set up config

In [3]:
# set up config
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2024-12-01"

In [4]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
dates_str_lst

['2023-01-01',
 '2023-02-01',
 '2023-03-01',
 '2023-04-01',
 '2023-05-01',
 '2023-06-01',
 '2023-07-01',
 '2023-08-01',
 '2023-09-01',
 '2023-10-01',
 '2023-11-01',
 '2023-12-01',
 '2024-01-01',
 '2024-02-01',
 '2024-03-01',
 '2024-04-01',
 '2024-05-01',
 '2024-06-01',
 '2024-07-01',
 '2024-08-01',
 '2024-09-01',
 '2024-10-01',
 '2024-11-01',
 '2024-12-01']

## Build Bronze Table

In [5]:
from utils.constants import BRONZE_FEAT_DIR

# run bronze backfill
for date_str in dates_str_lst:
    utils.data_processing_bronze_table.process_bronze_table(date_str, BRONZE_FEAT_DIR, spark)
    utils.data_processing_bronze_table.process_bronze_table_features(date_str, spark)

2023-01-01row count: 530
saved to: datamart/bronze/bronze_loan_daily_2023_01_01.csv
2023-01-01	row count: 530
saved to: datamart/bronze/bronze_features_attributes2023_01_01.csv
2023-01-01	row count: 530
saved to: datamart/bronze/bronze_features_financials2023_01_01.csv
2023-01-01	row count: 8974
saved to: datamart/bronze/bronze_feature_clickstream2023_01_01.csv
2023-02-01row count: 1031
saved to: datamart/bronze/bronze_loan_daily_2023_02_01.csv
2023-02-01	row count: 501
saved to: datamart/bronze/bronze_features_attributes2023_02_01.csv
2023-02-01	row count: 501
saved to: datamart/bronze/bronze_features_financials2023_02_01.csv
2023-02-01	row count: 8974
saved to: datamart/bronze/bronze_feature_clickstream2023_02_01.csv
2023-03-01row count: 1537
saved to: datamart/bronze/bronze_loan_daily_2023_03_01.csv
2023-03-01	row count: 506
saved to: datamart/bronze/bronze_features_attributes2023_03_01.csv
2023-03-01	row count: 506
saved to: datamart/bronze/bronze_features_financials2023_03_01.csv


## Build Silver Table

In [6]:
from utils.constants import SILVER_FEAT_DIR

# run silver backfill
for date_str in dates_str_lst:
    utils.data_processing_silver_table.process_silver_table(date_str, BRONZE_FEAT_DIR, SILVER_FEAT_DIR, spark)
    utils.data_processing_silver_table.process_silver_table_features(date_str, spark)

loaded from: datamart/bronze/bronze_loan_daily_2023_01_01.csv row count: 530
saved to: datamart/silver/silver_loan_daily_2023_01_01.parquet
loaded from: datamart/bronze/bronze_features_attributes2023_01_01.csv row count: 530


saved to: datamart/silver/silver_features_attributes2023_01_01.parquet
loaded from: datamart/bronze/bronze_features_financials2023_01_01.csv row count: 530
saved to: datamart/silver/silver_features_financials2023_01_01.parquet
loaded from: datamart/bronze/bronze_feature_clickstream2023_01_01.csv row count: 8974
saved to: datamart/silver/silver_feature_clickstream2023_01_01.parquet
loaded from: datamart/bronze/bronze_loan_daily_2023_02_01.csv row count: 1031
saved to: datamart/silver/silver_loan_daily_2023_02_01.parquet
loaded from: datamart/bronze/bronze_features_attributes2023_02_01.csv row count: 501
saved to: datamart/silver/silver_features_attributes2023_02_01.parquet
loaded from: datamart/bronze/bronze_features_financials2023_02_01.csv row count: 501
saved to: datamart/silver/silver_features_financials2023_02_01.parquet
loaded from: datamart/bronze/bronze_feature_clickstream2023_02_01.csv row count: 8974
saved to: datamart/silver/silver_feature_clickstream2023_02_01.parquet
loaded

## Build gold tables

In [7]:
from utils.constants import GOLD_FEAT_DIR

if not os.path.exists(GOLD_FEAT_DIR):
    os.makedirs(GOLD_FEAT_DIR)

# run gold backfill
for date_str in dates_str_lst:
    utils.data_processing_gold_table.process_labels_gold_table(date_str, SILVER_FEAT_DIR, GOLD_FEAT_DIR, spark, dpd = 30, mob = 6)
    utils.data_processing_gold_table.process_features_gold_table(date_str, spark)

loaded from: datamart/silver/silver_loan_daily_2023_01_01.parquet row count: 530
saved to: datamart/gold/feature_store/gold_label_store_2023_01_01.parquet
loaded from: datamart/silver/silver_features_attributes2023_01_01.parquet row count: 530
loaded from: datamart/silver/silver_features_financials2023_01_01.parquet row count: 530
loaded from: datamart/silver/silver_feature_clickstream2023_01_01.parquet row count: 8974
Final gold table row count after joins: 530
saved to: datamart/gold/feature_store/gold_feature_store_2023_01_01.parquet
loaded from: datamart/silver/silver_loan_daily_2023_02_01.parquet row count: 1031
saved to: datamart/gold/feature_store/gold_label_store_2023_02_01.parquet
loaded from: datamart/silver/silver_features_attributes2023_02_01.parquet row count: 501
loaded from: datamart/silver/silver_features_financials2023_02_01.parquet row count: 501
loaded from: datamart/silver/silver_feature_clickstream2023_02_01.parquet row count: 8974
Final gold table row count after 

## inspect label store

In [13]:
from utils.constants import GOLD_LABEL_DIR

files_list = [GOLD_LABEL_DIR+os.path.basename(f) for f in glob.glob(os.path.join(GOLD_LABEL_DIR, '*'))]
label_df = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",label_df.count())

label_df.show()

row_count: 8974
+--------------------+-----------+-----+----------+-------------+
|             loan_id|Customer_ID|label| label_def|snapshot_date|
+--------------------+-----------+-----+----------+-------------+
|CUS_0x1037_2023_0...| CUS_0x1037|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1069_2023_0...| CUS_0x1069|    0|30dpd_6mob|   2023-07-01|
|CUS_0x114a_2023_0...| CUS_0x114a|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1184_2023_0...| CUS_0x1184|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1297_2023_0...| CUS_0x1297|    1|30dpd_6mob|   2023-07-01|
|CUS_0x12fb_2023_0...| CUS_0x12fb|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1325_2023_0...| CUS_0x1325|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1341_2023_0...| CUS_0x1341|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1375_2023_0...| CUS_0x1375|    1|30dpd_6mob|   2023-07-01|
|CUS_0x13a8_2023_0...| CUS_0x13a8|    0|30dpd_6mob|   2023-07-01|
|CUS_0x13ef_2023_0...| CUS_0x13ef|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1440_2023_0...| CUS_0x1440|    0|30dpd_6mob|   2023-0

## Integrate feature and label tables ##

In [15]:
feature_files = [GOLD_FEAT_DIR + os.path.basename(f) 
                 for f in glob.glob(os.path.join(GOLD_FEAT_DIR, '*'))]
df_features = spark.read.parquet(*feature_files)
print("feature_store row_count:", df_features.count())

feature_store row_count: 8974


In [16]:
# This saves to GOLD_INTEGRATED_DIR + "gold_loans.parquet"
utils.data_processing_gold_table.feature_label_integration(df_features, label_df)

Intermediate gold_loans row count: 8974
Any nulls in dataframe: False
Final gold_loans row count: 8974


saved to: datamart/gold/gold_loans.parquet


DataFrame[Customer_ID: string, snapshot_date: string, Age: int, is_Scientist: int, is_Media_Manager: int, is_Musician: int, is_Lawyer: int, is_Teacher: int, is_Developer: int, is_Writer: int, is_Architect: int, is_Mechanic: int, is_Entrepreneur: int, is_Journalist: int, is_Doctor: int, is_Engineer: int, is_Accountant: int, is_Manager: int, Annual_Income: float, Monthly_Inhand_Salary: float, Num_Bank_Accounts: int, Num_Credit_Card: int, Interest_Rate: int, Num_of_Loan: int, Changed_Credit_Limit: float, Num_Credit_Inquiries: int, Credit_Utilization_Ratio: float, Total_EMI_per_month: float, Amount_invested_monthly: float, Monthly_Balance: float, has_mortgage_loan: int, has_personal_loan: int, has_home_equity_loan: int, has_payday_loan: int, has_debt_consolidation_loan: int, has_student_loan: int, has_credit_builder_loan: int, has_not_specified: int, has_auto_loan: int, Credit_Mix_Label: int, Credit_History_Months: int, Spending_Level: int, Payment_Value: int, fe_1: int, fe_2: int, fe_3: i

## Fit a LR model - ignore for grading

In [17]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from utils.constants import GOLD_INTEGRATED_DIR

gold_loans = spark.read.parquet(GOLD_INTEGRATED_DIR + "gold_loans.parquet")

# Drop non-feature columns
feature_cols = [col for col in gold_loans.columns if col not in ['Customer_ID', 'snapshot_date', 'loan_id', 'label', 'label_def']]

# Assemble features
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_unscaled", handleInvalid="keep")

# Scale features 
scaler = StandardScaler(inputCol="features_unscaled", outputCol="features", withStd=True, withMean=True)

# Initialize logistic regression
lr = LogisticRegression(featuresCol="features", labelCol="label", family="binomial")

# Create pipeline
pipeline = Pipeline(stages=[assembler, scaler, lr])

# Split data
train, test = gold_loans.randomSplit([0.7, 0.3], seed=88)

# Train model
model = pipeline.fit(train)

# Make predictions
predictions = model.transform(test)

# Evaluate
evaluator_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
evaluator_pr = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR")
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")

auc = evaluator_auc.evaluate(predictions)
pr = evaluator_pr.evaluate(predictions)
accuracy = evaluator_accuracy.evaluate(predictions)

print(f"Logistic Regression Results:")
print(f"AUC: {auc:.4f}")
print(f"PR: {pr:.4f}")
print(f"Accuracy: {accuracy:.4f}")

# Show coefficients (feature importance)
coefficients = model.stages[-1].coefficients
feature_importance = [(feature_cols[i], coefficients[i]) for i in range(len(feature_cols))]
feature_importance.sort(key=lambda x: abs(x[1]), reverse=True)

print("\nTop 10 Most Important Features:")
for feat, coef in feature_importance[:10]:
    print(f"  {feat}: {coef:.4f}")

# Show confusion matrix
predictions.groupBy("label", "prediction").count().show()

Logistic Regression Results:
AUC: 0.7787
PR: 0.6106
Accuracy: 0.7670

Top 10 Most Important Features:
  Interest_Rate: 0.5167
  Changed_Credit_Limit: -0.3334
  Spending_Level: -0.3179
  Credit_History_Months: -0.2806
  fe_10: -0.2769
  Num_Credit_Card: 0.2555
  fe_5: 0.2347
  Amount_invested_monthly: -0.2172
  fe_3: 0.1831
  fe_9: -0.1564
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0|  469|
|    0|       0.0| 1725|
|    1|       1.0|  332|
|    0|       1.0|  156|
+-----+----------+-----+

